#### Imports

In [1]:
import pandas as pd
import numpy as np
import time
import pyodbc
import pickle
import warnings
import os
from tensorflow.keras.models import load_model
from collections import deque
import winsound

# Suppress Deprecation and FutureWarnings for a cleaner live console output
warnings.filterwarnings('ignore')   

print("--- INITIALIZING FACTORY INFERENCE ENGINE ---")

--- INITIALIZING FACTORY INFERENCE ENGINE ---


#### Azure SQL & Path Configuration

In [2]:
# AZURE SQL CONFIGURATION
AZURE_SERVER = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM' 
AZURE_TABLE_NAME = 'MBP_ControllerData' 

# PATHS TO TRAINED ARTIFACTS
MODEL_PATH = 'best_overlock_lstm.keras'
SCALER_PATH = 'scaler.pkl'
ENCODER_PATH = 'encoder.pkl'

# Select Target Machine
SELECTED_MACHINE = input("Enter the Machine Serial Number to analyze (e.g., 0521760): ").strip()
print(f"Target Machine Set To: {SELECTED_MACHINE}")

# Connection String
driver = '{ODBC Driver 17 for SQL Server}'
conn_str = f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}'

Target Machine Set To: 0521760


#### Load Model & Maintenance Solutions

In [3]:
# MAINTENANCE KNOWLEDGE BASE
maintenance_solutions = {
    "Needle Breakages": "SOLUTION: Immediately replace the needle and check needle guard alignment.",
    "Thread Jamming": "SOLUTION: Clear the bobbin case, check thread tension, and clean feed dogs.",
    "Blade Broken": "SOLUTION: Schedule immediate replacement of upper/lower cutting blades.",
    "High Foot Pressure": "SOLUTION: Reduce presser foot pressure dial by 2 turns.",
    "Thread Breakages": "SOLUTION: Check thread quality and path; adjust tensioner settings."
}

try:
    model = load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    with open(ENCODER_PATH, 'rb') as f:
        encoder = pickle.load(f)
    print("✅ AI Model, Translators, and Solutions Loaded Successfully.")
except Exception as e:
    print(f"❌ ERROR LOADING FILES: {e}")

✅ AI Model, Translators, and Solutions Loaded Successfully.


#### Logic Engine: Live Feature Extraction(10Hz Banding)

In [4]:
def extract_live_features(df):
    df = df[df['machineVibration'].astype(str).str.startswith('10')].copy()
    if df.empty: return None

    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                f_end = f_start + 10
                vib_dict[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)
    
    vib_df = pd.DataFrame(vib_records)
    expected_cols = [f"{i}-{i+10}Hz" for i in range(10, 610, 10)]
    vib_df = vib_df.reindex(columns=expected_cols, fill_value=0)
    
    elec_cols = ['machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
                 'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax']
    elec_df = df[elec_cols].ffill().fillna(0)
    
    return pd.concat([vib_df, elec_df.reset_index(drop=True)], axis=1)

#### The Live Monitoring Loop

In [8]:
last_processed_time = None
TIME_STEPS = 5

try:
    print(f"🔄 Connecting to Azure SQL Database...")
    with pyodbc.connect(conn_str) as conn:
        print(f"✅ Monitoring Machine: {SELECTED_MACHINE}")
        query = f"SELECT TOP {TIME_STEPS} * FROM {AZURE_TABLE_NAME} WHERE machineSerialNumber = '{SELECTED_MACHINE}' ORDER BY dateTime DESC"
        
        while True:
            df_live = pd.read_sql(query, conn)
            if not df_live.empty and len(df_live) >= TIME_STEPS:
                db_raw_ts = df_live['dateTime'].iloc[0]
                
                if db_raw_ts != last_processed_time:
                    last_processed_time = db_raw_ts
                    
                    sl_record_time = pd.to_datetime(db_raw_ts) + pd.Timedelta(hours=5, minutes=30)
                    current_local_time = pd.Timestamp.now()
                    
                    df_window_raw = df_live.iloc[::-1].reset_index(drop=True)
                    df_features = extract_live_features(df_window_raw)
                    
                    if df_features is not None and len(df_features) == TIME_STEPS:
                        X_scaled = scaler.transform(df_features.values).reshape(1, TIME_STEPS, -1)
                        probs = model.predict(X_scaled, verbose=0)[0]
                        max_idx = np.argmax(probs)
                        confidence = probs[max_idx]
                        predicted_state = encoder.inverse_transform([max_idx])[0]
                        
                        print("\n" + "="*45)
                        print(f"AZURE RECORD (UTC) : {str(db_raw_ts).split('.')[0]}") 
                        print(f"RECORD TIME (SL)   : {sl_record_time.strftime('%Y-%m-%d %H:%M:%S')}")
                        print(f"CURRENT LOCAL TIME : {current_local_time.strftime('%Y-%m-%d %H:%M:%S')}")
                        print("-" * 45)
                        print(f"PREDICTED STATE    : {predicted_state} ({confidence*100:.2f}%)")
                        
                        if predicted_state != "Healthy" and confidence >= 0.75:
                            print(f"🚨 ALERT: {predicted_state} DETECTED")
                            winsound.Beep(1000, 500)
                            winsound.Beep(1000, 500)
                            winsound.Beep(1000, 500)
                            # Restoration of the Solution logic
                            if predicted_state in maintenance_solutions:
                                print(f"🛠️ {maintenance_solutions[predicted_state]}")
                        elif confidence < 0.75:
                            print("⚠️ UNKNOWN STATE: Anomaly suspected")
                            winsound.Beep(1000, 500)
                            winsound.Beep(1000, 500)
                        else:
                            print("✅ STATUS: Optimal Health")
                        print("="*45)
            
            time.sleep(1)

except KeyboardInterrupt:
    print("\n🛑 Stopped by user.")
except Exception as e:
    print(f"\n❌ SCRIPT ERROR: {e}")

🔄 Connecting to Azure SQL Database...
✅ Monitoring Machine: 0521760

AZURE RECORD (UTC) : 2026-03-10 09:20:41
RECORD TIME (SL)   : 2026-03-10 14:50:41
CURRENT LOCAL TIME : 2026-03-10 22:17:41
---------------------------------------------
PREDICTED STATE    : Healthy (15.70%)
⚠️ UNKNOWN STATE: Anomaly suspected

❌ SCRIPT ERROR: Execution failed on sql: SELECT TOP 5 * FROM MBP_ControllerData WHERE machineSerialNumber = '0521760' ORDER BY dateTime DESC
('08S01', '[08S01] [Microsoft][ODBC Driver 17 for SQL Server]TCP Provider: An established connection was aborted by the software in your host machine.\r\n (10053) (SQLExecDirectW); [08S01] [Microsoft][ODBC Driver 17 for SQL Server]Communication link failure (10053)')
unable to rollback
